In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.evt import fit_pot, evt_threshold
from src.evaluation import tpr_at_threshold

SC = ROOT / "results" / "scores"
files = {"M3 AE v2 (max)": "w_m3v2_max_seed11.parquet",
         "M4 TCN": "w_m4_seed11.parquet",
         "M4 + M3v2 (max)": "w_fused_max_seed11.parquet"}
QS = [1e-2, 1e-3]

rows, diag = [], []
for name, f in files.items():
    df = pd.read_parquet(SC / f)
    val = df[(df["split"] == "val") & (df["label"] == 0)]["score"].to_numpy()
    fit = fit_pot(val)
    diag.append({"model": name, **{k: round(v, 4) for k, v in fit.items()}})
    for q in QS:
        for method, thr in [("EVT", evt_threshold(fit, q)),
                            ("percentile", float(np.quantile(val, 1 - q)))]:
            for sp in ["tune", "test"]:
                d = df[df["split"] == sp]
                tpr, fpr = tpr_at_threshold(d, thr)
                macro = np.mean([(g["score"] > thr).mean()
                                 for _, g in d[d["label"] == 1].groupby("attack_type")])
                rows.append({"model": name, "target_fpr": q, "method": method, "split": sp,
                             "real_fpr": fpr, "tpr": tpr, "macro_tpr": macro})

print(pd.DataFrame(diag).to_string(index=False))
evt_w = pd.DataFrame(rows).round(4)
evt_w.to_csv(ROOT / "results" / "evt_windows.csv", index=False)
evt_w[evt_w["split"] == "tune"]

          model      t      xi  sigma     n  n_t  ks_stat   ks_p
 M3 AE v2 (max) 0.0612  0.4922 0.0677 30000 1500   0.0316 0.0971
         M4 TCN 1.0826  0.1105 0.7717 30000 1500   0.0199 0.5880
M4 + M3v2 (max) 2.2307 -0.1829 0.4217 30000 1500   0.0430 0.0075


,model,target_fpr,method,split,real_fpr,tpr,macro_tpr
0,M3 AE v2 (max),0.010,EVT,tune,0.0080,0.0176,0.0384
2,M3 AE v2 (max),0.010,percentile,tune,0.0080,0.0176,0.0384
4,M3 AE v2 (max),0.001,EVT,tune,0.0006,0.0023,0.0047
6,M3 AE v2 (max),0.001,percentile,tune,0.0005,0.0022,0.0045
8,M4 TCN,0.010,EVT,tune,0.0509,0.1388,0.2825
10,M4 TCN,0.010,percentile,tune,0.0501,0.1366,0.2792
12,M4 TCN,0.001,EVT,tune,0.0013,0.0126,0.0458
14,M4 TCN,0.001,percentile,tune,0.0018,0.0139,0.0503
16,M4 + M3v2 (max),0.010,EVT,tune,0.0082,0.0260,0.0725
18,M4 + M3v2 (max),0.010,percentile,tune,0.0085,0.0270,0.0753
